# Regulatory Outbound Design Analysis

## tl;dr

This notebook converts Digital Asset regulatory analysis into outbound handoff design artifacts. It excludes synthetic policy simulation from main design evidence.

## Context & Methods

Inputs are regulatory raw obligations, Ethereum transaction field mapping v3, outbound requirement CSV, execution mapping, and runtime pipeline artifacts.

### Key Assumptions

FPG is a pre-external-handoff policy enforcement gateway. It does not approve transactions, perform KYC/AML, custody assets, sign transactions, execute smart contracts, or guarantee settlement finality.

In [ ]:
# ruff: noqa: E501, E701, E702, I001

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name != 'ADP-DA': ROOT = next(p for p in [ROOT, *ROOT.parents] if p.name == 'ADP-DA')
req = pd.read_csv(ROOT/'03_digital_asset/data/processed/regulatory_outbound_requirements.csv')
exec_map = pd.read_csv(ROOT/'03_digital_asset/data/processed/outbound_requirement_execution_mapping.csv')
matrix = json.loads((ROOT/'03_digital_asset/artifacts/outbound_design_v1/outbound_requirement_matrix.json').read_text(encoding='utf-8'))
pipeline = json.loads((ROOT/'03_digital_asset/artifacts/outbound_design_v1/runtime_pipeline.json').read_text(encoding='utf-8'))
print({'requirements': len(req), 'pipeline_steps': len(pipeline['steps']), 'matrix_requirements': len(matrix['requirements'])})

## Data

### 1. Legal Obligation Distribution

In [ ]:
display(req.groupby(['evidence_type','effective_status']).size().reset_index(name='count'))

## Results

### 2. Outbound Requirement Type Distribution

In [ ]:
counts = req['outbound_requirement_type'].value_counts().reset_index(); counts.columns=['outbound_requirement_type','count']; display(counts); counts.plot.bar(x='outbound_requirement_type', y='count', legend=False, color='#2f6f8f', title='Outbound Requirement Type Distribution'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 3. Required Field x Source System

In [ ]:
src = req.assign(source_system=req['source_system'].str.split('|')).explode('source_system')
display(src.groupby(['required_field','source_system']).size().unstack(fill_value=0))

### 4. Required Field x Destination

In [ ]:
dest = req.assign(destination=req['destination'].str.split('|')).explode('destination')
display(dest.groupby(['required_field','destination']).size().unstack(fill_value=0))

### 5. Required Field x Transform

In [ ]:
tr = req.assign(transform_type=req['transform_type'].str.split('|')).explode('transform_type')
display(tr.groupby(['required_field','transform_type']).size().unstack(fill_value=0))

### 6. On-chain vs Off-chain Distribution

In [ ]:
layers = exec_map['availability_layer'].value_counts().reset_index(); layers.columns=['availability_layer','count']; display(layers); layers.plot.bar(x='availability_layer', y='count', legend=False, color='#4f8f65', title='On-chain vs Off-chain Requirement Mapping'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()

### 7. Mandatory vs Conditional

In [ ]:
display(req['mandatory_level'].value_counts().rename_axis('mandatory_level').reset_index(name='count'))

### 8. Required Exact Ratio

In [ ]:
exact_ratio = req['required_exact'].mean() if req['required_exact'].dtype == bool else (req['required_exact'].astype(str) == 'TRUE').mean(); print({'required_exact_ratio': round(exact_ratio, 4)})

### 9. Externalized vs Internal-only

In [ ]:
destination_flags = dest.assign(externalized=dest['destination'].isin(['EXTERNAL_VASP','TRAVEL_RULE_PROVIDER','BLOCKCHAIN_EXECUTION_SYSTEM'])); display(destination_flags.groupby('externalized').size().reset_index(name='field_destination_rows'))

### 10. Control -> Requirement -> Field -> Destination Lineage

In [ ]:
display(req[['requirement_id','obligation_id','outbound_requirement_type','required_field','source_system','destination','transform_type','evidence_type']])

## Takeaways

The design separates legal requirement, internal approval requirement, execution requirement, and privacy requirement. BE can implement the handoff using versioned JSON artifacts, but BE-owned runtime enums and provider schemas remain contract gaps.